<a href="https://colab.research.google.com/github/hannaginther/ENGG680_2025_Fall/blob/Allison/MachineLearningTest.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from google.colab import drive
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

drive.mount('/content/drive')

#Loading Data
path = "/content/drive/MyDrive/sparcs_model_v1.feather"
data = pd.read_feather(path)



Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
#Looking at the Dataset Info
print(data.shape)
data.info()
data.head()

(4238636, 33)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4238636 entries, 0 to 4238635
Data columns (total 33 columns):
 #   Column                   Dtype  
---  ------                   -----  
 0   health_service_area      object 
 1   hospital_county          object 
 2   facility_id              object 
 3   age_group                object 
 4   zip_code                 object 
 5   gender                   object 
 6   race                     object 
 7   ethnicity                object 
 8   length_of_stay           int64  
 9   admission_type           object 
 10  disposition              object 
 11  discharge_year           int64  
 12  ccsr_dx_code             object 
 13  ccsr_px_code             object 
 14  apr_drg_code             object 
 15  apr_mdc_code             object 
 16  apr_severity_code        object 
 17  apr_mortality_risk       object 
 18  apr_med_surg_desc        object 
 19  total_charges            float64
 20  payer_medicaid           int64  

,health_service_area,hospital_county,facility_id,age_group,zip_code,gender,race,ethnicity,length_of_stay,admission_type,...,payer_self_pay,payer_blue_cross,payer_other,payer_gov_va,payer_corrections,payer_managed_care,num_payment_types,log_total_charges,los_log,los_yj
0,New York City,Bronx,3058,50-69,104,F,Other Race,Spanish/Hispanic,1,Emergency,...,0,0,0,0,0,0,1,10.217627,0.693147,-1.501001
1,New York City,Bronx,1168,30-49,104,M,Black/African American,Not Span/Hispanic,4,Emergency,...,0,0,0,0,0,0,1,11.322140,1.609438,0.246954
2,New York City,Bronx,3058,50-69,104,M,Other Race,Not Span/Hispanic,4,Emergency,...,0,0,0,0,0,0,2,11.241267,1.609438,0.246954
3,New York City,Bronx,1169,18-29,104,M,Black/African American,Not Span/Hispanic,5,Emergency,...,0,0,0,0,0,0,1,11.258467,1.791759,0.503353
4,New York City,Bronx,1169,50-69,104,F,Other Race,Spanish/Hispanic,3,Emergency,...,0,0,0,0,0,0,2,11.057713,1.386294,-0.103046


In [ ]:
drop_columns=['disposition', 'discharge_year', 'ccsr_px_cod', 'apr_med_surg_desc']

In [ ]:
numerical_columns = ['payer_medicaid', 'payer_medicare','payer_private_insurance','payer_self_pay',
                     'payer_blue_cross','payer_other', 'payer_gov_va',
                     'payer_corrections','payer_managed_care','num_payment_types']


categorical_cols = ['zip_code', 'health_service_area', 'facility_id', 'hospital_county', 'age_group',
                    'gender', 'race', 'ethnicity', 'admission_type', 'apr_mortality_risk',
                    'apr_severity_code', 'apr_drg_code', 'apr_mdc_code', 'ccsr_dx_code']

In [ ]:
#Creating a function to clean and prepare data set
def clean_data(df):

#Remove Duplicates
  df = df.drop_duplicates()

#Excluding the rows that do not have LOS; Missing data
  df = df.dropna(subset = ['length_of_stay'])

#Convert data to numerical, Drop those who can't convert
  df['length_of_stay'] = pd.to_numeric(df['length_of_stay'], errors='coerce')
  df = df.dropna(subset = ['length_of_stay'])

#Remove out-of-state ZIP Codes ('OOS')
  if 'Zip Code' in df.columns:
      df = df[df['zip_code'] != 'OOS']

  df = df.drop(columns =[c for c in drop_columns if c in df.columns], errors = 'ignore')

  return df

#Clean Data
data = clean_data(data)

print("Shape of cleaned data:", data.shape)  # rows, columns




Shape of cleaned data: (4206911, 29)


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from lightgbm import LGBMRegressor
import re

#Setting up parameters,
X = data.drop(columns = ['length_of_stay']) # Data Features, everything but LOS
y = data['length_of_stay'] #Target Variable

categorical_columns = X.select_dtypes(include=['object']).columns.tolist()

X = pd.get_dummies(X, columns=categorical_columns, drop_first=True)

# ---- FIX: Clean feature names to avoid LightGBM JSON errors ----
X.columns = [
    re.sub(r'[^A-Za-z0-9_]+', '_', col).strip('_')
    for col in X.columns
]

# Handle any duplicates that may result from cleaning
if X.columns.duplicated().any():
    unique_cols = []
    for col in X.columns:
        if col not in unique_cols:
            unique_cols.append(col)
        else:
            i = 1
            new_col = f"{col}_{i}"
            while new_col in unique_cols:
                i += 1
                new_col = f"{col}_{i}"
            unique_cols.append(new_col)
    X.columns = unique_cols

#80/20 split test
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.3, random_state =42)

#Training model
model = LGBMRegressor(n_estimators=300, learning_rate=0.05, num_leaves=64)
model.fit(X_train, y_train)

#Predictions
y_pred = model.predict(X_test)

importances = pd.Series(model.feature_importances_, index=X.columns).sort_values(ascending=False)

plt.figure(figsize=(10,6))
importances.head(15).plot(kind='barh', color='skyblue')
plt.gca().invert_yaxis()
plt.title('Top 15 Feature Importances for Length of Stay Prediction')
plt.xlabel('Importance Score')
plt.ylabel('Features')
plt.show()

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

#Plot Outliers
plt.figure(figsize = (10,5))
sns.boxplot(x = data['length_of_stay'])
plt.title('Potential Outliers on Length of Stay')
plt.show()

plt.figure(figsize = (8,5))
sns.histplot(data['length_of_stay'], bins = 40, kde = True)
plt.title('Distribution of Length of Stay')
plt.xlabel('Days')
plt.show()

data['length_of_stay'].describe()

In [ ]:
#Setting up parameters,
X = data.drop(columns = ['length_of_stay']) # Data Features, everything but LOS
y = data['length_of_stay'] #Target Variable

categorical_columns = X.select_dtypes(include=['object']).columns.tolist()
categorical_columns

X = pd.get_dummies(X, columns=categorical_columns, drop_first=True)

#80/20 split test
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.3, random_state =42)

#Training model
model = RandomForestRegressor(n_estimators = 300, random_state=42, max_depth = None, min_samples_split = 2)
model.fit(X_train, y_train)

#Predictions
y_pred = model.predict(X_test)

#Evaluating
mae = mean_absolute_error(y_test, y_pred) #The average absolute prediction error (in days)
rmse = (mean_squared_error(y_test, y_pred))**0.5
r2 = r2_score(y_test, y_pred)

print("RMSE:", rmse)
print("R²:", r2)
print(f'Mean Absolute Error: {mae: .2f}')